In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score

In [8]:
train_sample_path = "https://raw.githubusercontent.com/dinanabila/hackathon-linear-regression-yandex-2026/refs/heads/main/train_sample.csv?token=GHSAT0AAAAAAEF7G4WIQ6232OZ42E5YWVR62U2A5ZQ"
test_sample_path = "https://raw.githubusercontent.com/dinanabila/hackathon-linear-regression-yandex-2026/refs/heads/main/test_sample.csv?token=GHSAT0AAAAAAEF7G4WIGNEBHNWEVSJB4ZZS2U2A6HA"

In [9]:
train_sample = pd.read_csv(train_sample_path)
train_sample.head()

,start_point,end_point,time_of_day,day_of_week,traffic_condition,event_count,is_holiday,vehicle_density,population_density,weather,public_transport_availability,historical_delay_factor,travel_time
0,West Jakarta (Jakarta Barat),South Jakarta (Jakarta Selatan),day,Sunday,NaN,9,1,NaN,high,NaN,1,0.878909,26.907612
1,West Jakarta (Jakarta Barat),South Jakarta (Jakarta Selatan),morning,Thursday,NaN,7,1,medium,high,NaN,1,1.081668,27.489129
2,Central Jakarta (Jakarta Pusat),East Jakarta (Jakarta Timur),morning,Thursday,NaN,7,0,medium,low,NaN,2,1.192379,27.228978
3,West Jakarta (Jakarta Barat),South Jakarta (Jakarta Selatan),morning,Friday,10.0,9,0,medium,high,fog,1,0.833348,33.943970
4,Central Jakarta (Jakarta Pusat),West Jakarta (Jakarta Barat),day,Tuesday,NaN,7,0,medium,high,rain,2,0.966819,20.603115


In [10]:
test_sample = pd.read_csv(test_sample_path)
test_sample.head(3)

,start_point,end_point,time_of_day,day_of_week,traffic_condition,event_count,is_holiday,vehicle_density,population_density,weather,public_transport_availability,historical_delay_factor
0,West Jakarta (Jakarta Barat),East Jakarta (Jakarta Timur),morning,Saturday,5.0,8,1,medium,NaN,NaN,2,1.126429
1,South Jakarta (Jakarta Selatan),East Jakarta (Jakarta Timur),evening,Saturday,NaN,9,1,low,medium,fog,2,1.121015
2,Central Jakarta (Jakarta Pusat),South Jakarta (Jakarta Selatan),morning,Friday,9.0,8,0,high,NaN,rain,1,1.109638


In [11]:
train_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 13 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   start_point                    40000 non-null  str    
 1   end_point                      40000 non-null  str    
 2   time_of_day                    40000 non-null  str    
 3   day_of_week                    40000 non-null  str    
 4   traffic_condition              25599 non-null  float64
 5   event_count                    40000 non-null  int64  
 6   is_holiday                     40000 non-null  int64  
 7   vehicle_density                25622 non-null  str    
 8   population_density             25552 non-null  str    
 9   weather                        25571 non-null  str    
 10  public_transport_availability  40000 non-null  int64  
 11  historical_delay_factor        40000 non-null  float64
 12  travel_time                    40000 non-null  float64
dt

In [12]:
train_sample.describe()

,traffic_condition,event_count,is_holiday,public_transport_availability,historical_delay_factor,travel_time
count,25599.000000,40000.000000,40000.000000,40000.000000,40000.000000,40000.000000
mean,8.455877,8.704900,0.500975,1.499525,1.009510,30.405411
std,1.829816,5.374108,0.500005,0.671201,0.162034,15.074854
min,3.000000,7.000000,0.000000,0.000000,0.750009,2.490207
25%,9.000000,7.000000,0.000000,1.000000,0.878415,18.685606
50%,9.000000,8.000000,1.000000,2.000000,1.003418,26.604213
75%,10.000000,9.000000,1.000000,2.000000,1.132044,39.771383
max,10.000000,88.000000,1.000000,2.000000,2.701010,218.832465


In [13]:
train_sample.isnull().sum()

start_point                          0
end_point                            0
time_of_day                          0
day_of_week                          0
traffic_condition                14401
event_count                          0
is_holiday                           0
vehicle_density                  14378
population_density               14448
weather                          14429
public_transport_availability        0
historical_delay_factor              0
travel_time                          0
dtype: int64

In [14]:
train_sample.duplicated().sum()

np.int64(0)

In [21]:
# Numeric → mean
numeric_cols = train_sample.select_dtypes(include='number').columns

train_sample[numeric_cols] = train_sample[numeric_cols].fillna(
    train_sample[numeric_cols].mean()
)

# Categorical → mode
categorical_cols = train_sample.select_dtypes(
    include=['str']
).columns

for col in categorical_cols:
    train_sample[col] = train_sample[col].fillna(
        train_sample[col].mode()[0]
    )

train_sample.isnull().sum()

start_point                      0
end_point                        0
time_of_day                      0
day_of_week                      0
traffic_condition                0
event_count                      0
is_holiday                       0
vehicle_density                  0
population_density               0
weather                          0
public_transport_availability    0
historical_delay_factor          0
travel_time                      0
dtype: int64

In [22]:
test_sample.isnull().sum()

start_point                        0
end_point                          0
time_of_day                        0
day_of_week                        0
traffic_condition                600
event_count                        0
is_holiday                         0
vehicle_density                  600
population_density               600
weather                          600
public_transport_availability      0
historical_delay_factor            0
dtype: int64

In [23]:
# Numeric → mean
numeric_cols = test_sample.select_dtypes(include='number').columns

test_sample[numeric_cols] = test_sample[numeric_cols].fillna(
    test_sample[numeric_cols].mean()
)

# Categorical → mode
categorical_cols = test_sample.select_dtypes(
    include=['str']
).columns

for col in categorical_cols:
    test_sample[col] = test_sample[col].fillna(
        test_sample[col].mode()[0]
    )

test_sample.isnull().sum()

start_point                      0
end_point                        0
time_of_day                      0
day_of_week                      0
traffic_condition                0
event_count                      0
is_holiday                       0
vehicle_density                  0
population_density               0
weather                          0
public_transport_availability    0
historical_delay_factor          0
dtype: int64

In [24]:
train_sample.drop_duplicates(inplace=True)
train_sample.duplicated().sum()

np.int64(0)

(opsional) drop column

In [ ]:
# cols_to_drop = ['id', 'date']
# train_sample = train_sample.drop(columns=cols_to_drop)
# train_sample.head(2)

In [ ]:
# test_sample = test_sample.drop(columns=cols_to_drop)
# test_sample.head(2)

train val split

In [28]:
X_train, X_val, y_train, y_val = train_test_split(train_sample.drop(columns=['travel_time']), train_sample['travel_time'], test_size=0.2, random_state=38)

In [29]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_train index:", X_train.index[:5])
print("y_train index:", y_train.index[:5])

X_train: (32000, 12)
y_train: (32000,)
X_train index: Index([10777, 35199, 38951, 3007, 31110], dtype='int64')
y_train index: Index([10777, 35199, 38951, 3007, 31110], dtype='int64')


one-hot encoding

In [ ]:
# ohe = OneHotEncoder(sparse=False, drop='first', handle_unknown='ignore')
# X_train_encoded = ohe.fit_transform(X_train)
# X_val_encoded = ohe.transform(X_val)
# X_test_encoded = ohe.transform(test_sample)
# X_train_encoded.head(3)

In [30]:
from sklearn.model_selection import cross_validate, KFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols),
        ('num', StandardScaler(), numeric_cols)
    ]
)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=['r2', 'neg_mean_squared_error', 'neg_mean_absolute_error']
)

print('R²:', scores['test_r2'].mean())
print('MAE:', -scores['test_neg_mean_absolute_error'].mean())
print('MSE:', -scores['test_neg_mean_squared_error'].mean())

R²: 0.8804750918554815
MAE: 3.435093469774987
MSE: 27.083737285915618


In [33]:
model.fit(X_train, y_train)

y_pred = model.predict(test_sample)

pd.DataFrame(y_pred).to_csv('submission.csv', index=False)

In [34]:
pd.read_csv('submission.csv')['0']

0       45.837479
1       21.657716
2       24.299142
3       20.562907
4       36.187265
          ...    
2995    41.618348
2996    24.441042
2997    16.145035
2998    13.592968
2999    16.005269
Name: 0, Length: 3000, dtype: float64